### **1. Import, Load, Clean**

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
# Load dataset
PATH = "../data/raw/listings.csv"
df = pd.read_csv(PATH)

# Price cleaning
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Percentage cleaning
rates = ["host_response_rate", "host_acceptance_rate"]
for col in rates:
    df[col] = df[col].astype(str).str.replace("%", "").astype(float) / 100

# Remove rows where price is missing
df = df.dropna(subset=["price"]).copy()

In [3]:
cols_to_drop = [

    # identifiers / urls / metadata
    "id",
    "listing_url",
    "scrape_id",
    "last_scraped",
    "source",
    "picture_url",
    "host_id",
    "host_url",
    "host_thumbnail_url",
    "host_picture_url",
    "calendar_updated",
    "calendar_last_scraped",

    # leakage
    "estimated_revenue_l365d",
    "estimated_occupancy_l365d",

    # text fields (no NLP)
    "name",
    "description",
    "neighborhood_overview",
    "host_about",

    # redundant text versions
    "bathrooms_text",
    "host_name",
    "host_verifications",

    # review score redundancy
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",

    # review redundancy
    "number_of_reviews",

    # host listing redundancy
    "host_listings_count",
    "host_total_listings_count",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms",

    # availability redundancy
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_eoy",

    # review activity redundancy
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",

    # derived night statistics
    "minimum_minimum_nights",
    "maximum_minimum_nights",
    "minimum_maximum_nights",
    "maximum_maximum_nights",
    "minimum_nights_avg_ntm",
    "maximum_nights_avg_ntm",

    # categorical removal from EDA
    "host_since",
    "first_review",
    "last_review",
    "amenities",
    "license",
    "host_location",
    "host_neighbourhood",
    "neighbourhood",
    "neighbourhood_cleansed",

    # weak categorical predictor
    "host_response_time",
    "host_response_rate",
    "host_acceptance_rate"
]

In [4]:
df["price_bin"] = pd.qcut(df["price"], q=5, duplicates="drop")

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["price_bin"]
)

df_train = df_train.drop(columns=["price_bin"])
df_test = df_test.drop(columns=["price_bin"])

In [5]:
df_train["log_price"] = np.log1p(df_train["price"])
df_test["log_price"] = np.log1p(df_test["price"])

In [6]:
X_train = df_train.drop(["price", "log_price"], axis=1)
y_train = df_train["log_price"]

X_test = df_test.drop(["price", "log_price"], axis=1)
y_test = df_test["log_price"]

In [7]:
X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
X_test = X_test.drop(columns=cols_to_drop, errors="ignore")

In [8]:
print(X_train.shape)
print(X_test.shape)

(4520, 20)
(1131, 20)


In [10]:
X_train.columns

Index(['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
       'neighbourhood_group_cleansed', 'latitude', 'longitude',
       'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms',
       'beds', 'minimum_nights', 'maximum_nights', 'has_availability',
       'availability_365', 'review_scores_rating', 'instant_bookable',
       'calculated_host_listings_count', 'reviews_per_month'],
      dtype='str')

In [9]:
X_train.describe()

,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,availability_365,review_scores_rating,calculated_host_listings_count,reviews_per_month
count,4520.000000,4520.000000,4520.000000,4516.000000,4518.000000,4509.000000,4520.000000,4.520000e+03,4520.000000,4098.000000,4520.000000,4098.000000
mean,43.260544,-2.510491,4.043805,1.518379,1.992917,3.017964,3.103540,2.290641e+04,203.566593,4.749219,11.655088,1.549048
std,0.151415,0.435468,2.342981,1.019356,1.271072,2.362151,7.113805,1.487473e+06,119.513660,0.322975,26.532462,1.829270
min,42.487640,-3.378762,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000e+00,0.000000,1.000000,1.000000,0.010000
25%,43.257148,-2.925680,2.000000,1.000000,1.000000,1.000000,1.000000,1.800000e+02,86.000000,4.670000,1.000000,0.440000
50%,43.299497,-2.670280,4.000000,1.000000,2.000000,3.000000,2.000000,3.650000e+02,219.500000,4.830000,2.000000,1.000000
75%,43.323660,-1.985796,5.000000,2.000000,3.000000,4.000000,2.000000,1.125000e+03,322.000000,4.940000,8.000000,2.110000
max,43.441490,-1.757010,16.000000,24.000000,25.000000,48.000000,160.000000,1.000000e+08,365.000000,5.000000,149.000000,47.610000


### **2. Handling missing values**

In [ ]:
missing = X_train.isna().mean().mul(100).sort_values(ascending=False)
missing[missing > 0]

In [ ]:
from sklearn.impute import SimpleImputer

num_cols = X_train.select_dtypes(include="number").columns

num_imputer = SimpleImputer(strategy="median")

X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

In [ ]:
cat_cols = X_train.select_dtypes(include="object").columns

cat_imputer = SimpleImputer(strategy="most_frequent")

X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

In [ ]:
X_train.isna().sum().sum(), X_test.isna().sum().sum()

### **3. Transformations**

In [ ]:
BILBAO_LAT = 43.2630
BILBAO_LON = -2.9350

def distance_to_bilbao(lat, lon):
    return np.sqrt((lat - BILBAO_LAT)**2 + (lon - BILBAO_LON)**2)

X_train["distance_to_bilbao"] = distance_to_bilbao(
    X_train["latitude"], X_train["longitude"]
)

X_test["distance_to_bilbao"] = distance_to_bilbao(
    X_test["latitude"], X_test["longitude"]
)

In [ ]:
DONOSTIA_LAT = 43.3183
DONOSTIA_LON = -1.9812

def distance_to_donostia(lat, lon):
    return np.sqrt((lat - DONOSTIA_LAT)**2 + (lon - DONOSTIA_LON)**2)

X_train["distance_to_donostia"] = distance_to_donostia(
    X_train["latitude"], X_train["longitude"]
)

X_test["distance_to_donostia"] = distance_to_donostia(
    X_test["latitude"], X_test["longitude"]
)

In [ ]:
VITORIA_LAT = 42.8467
VITORIA_LON = -2.6726

def distance_to_vitoria(lat, lon):
    return np.sqrt((lat - VITORIA_LAT)**2 + (lon - VITORIA_LON)**2)

X_train["distance_to_vitoria"] = distance_to_vitoria(
    X_train["latitude"], X_train["longitude"]
)

X_test["distance_to_vitoria"] = distance_to_vitoria(
    X_test["latitude"], X_test["longitude"]
)

In [ ]:
COAST_LAT = 43.3623
COAST_LON = -3.0136

def distance_to_coast(lat, lon):
    return np.sqrt((lat - COAST_LAT)**2 + (lon - COAST_LON)**2)

X_train["distance_to_coast"] = distance_to_coast(
    X_train["latitude"], X_train["longitude"]
)

X_test["distance_to_coast"] = distance_to_coast(
    X_test["latitude"], X_test["longitude"]
)

In [ ]:
X_train = X_train.drop(columns=["latitude", "longitude"])
X_test = X_test.drop(columns=["latitude", "longitude"])

In [ ]:
clip_cols = [
    "beds",
    "minimum_nights",
    "maximum_nights"
]

for col in clip_cols:
    
    lower = X_train[col].quantile(0.01)
    upper = X_train[col].quantile(0.99)

    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)

In [ ]:
skewed = [
    "minimum_nights",
    "maximum_nights",
    "reviews_per_month",
    "calculated_host_listings_count"
]

for col in skewed:
    X_train[col] = np.log1p(X_train[col])
    X_test[col] = np.log1p(X_test[col])

In [ ]:
cat_cols = X_train.select_dtypes(include="object").columns
cat_cols

In [ ]:
binary_cols = [
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "has_availability",
    "instant_bookable"
]

for col in binary_cols:
    X_train[col] = X_train[col].map({"t":1, "f":0})
    X_test[col] = X_test[col].map({"t":1, "f":0})

In [ ]:
TOP_K = 10

top_properties = (
    X_train["property_type"]
    .value_counts()
    .nlargest(TOP_K)
    .index
)

X_train["property_type_clean"] = X_train["property_type"].where(
    X_train["property_type"].isin(top_properties),
    "Other"
)

X_test["property_type_clean"] = X_test["property_type"].where(
    X_test["property_type"].isin(top_properties),
    "Other"
)

X_train = X_train.drop(columns=["property_type"])
X_test = X_test.drop(columns=["property_type"])

In [ ]:
categorical_cols = [
    "property_type_clean",
    "room_type",
    "neighbourhood_group_cleansed"
]

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

### **4. Model**

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import RidgeCV
import numpy as np

alphas = np.logspace(-3, 3, 50)

ridge_cv = RidgeCV(
    alphas=alphas,
    scoring="neg_root_mean_squared_error",
    cv=5
)

ridge_cv.fit(X_train_scaled, y_train)

print("Best alpha:", ridge_cv.alpha_)

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=ridge_cv.alpha_)

ridge.fit(X_train_scaled, y_train)

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    ridge,
    X_train_scaled,
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=5
)

rmse = -scores.mean()

print("CV RMSE:", rmse)